# 📊 Professional Exploratory Data Analysis

This notebook is a **professional EDA notebook** for the Marketing Intelligence Platform.

It keeps the full chart and analysis set, but the code is written for learning:


## What this notebook analyzes

This notebook reads the cleaned order-level table:

```text
marketing.db → master_data
```

The ETL pipeline already created this table.

Important data grain:

> **One row = one order**

That means revenue and order KPIs are safer because payments and items were aggregated before merging.


# 0. How to Run This Notebook

From your project root, run:

```bash
source venv/bin/activate
python main.py
jupyter notebook
```

If Plotly does not render charts, install:

```bash
source venv/bin/activate
python -m pip install plotly nbformat ipykernel
```

---


In [1]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

# pandas is used for tabular data analysis.
# In this notebook, almost every calculation starts with a pandas dataframe.
import pandas as pd

# numpy is used for numeric operations.
# We mainly use it for safe missing-value handling.
import numpy as np

# Path helps us work with file locations in a safe way.
# It avoids hardcoding long file paths.
from pathlib import Path

# create_engine creates a database connection object.
# pandas uses this engine to read data from SQLite.
from sqlalchemy import create_engine

# plotly.express creates professional charts with relatively simple code.
# We use it for bar charts, histograms, scatter plots, treemaps, and heatmaps.
import plotly.express as px

# plotly.graph_objects gives more control than plotly.express.
# We use it when we need combined charts such as bar + line charts.
import plotly.graph_objects as go

# display and Markdown allow us to show formatted text inside the notebook.
# We use them for insight and conclusion boxes.
from IPython.display import display, Markdown

# This setting shows more columns when displaying dataframes.
# Without this, pandas may hide useful columns.
pd.set_option("display.max_columns", 100)

# This setting makes large decimal numbers easier to read.
# Example: 1234567.891 becomes 1,234,567.89
pd.options.display.float_format = "{:,.2f}".format


# 1. Helper Functions

Helper functions are small reusable blocks of code.

Why we use them here:

- avoid repeating formatting code
- keep chart style consistent
- make the notebook easier to read
- display conclusions in a clean format


In [2]:
# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def format_currency(value):
    """Convert large revenue numbers into readable text."""

    # If value is missing, return 0 as text.
    if pd.isna(value):
        return "0"

    # If value is at least one million, show it in millions.
    # Example: 2,500,000 becomes 2.50M.
    if abs(value) >= 1_000_000:
        return f"{value / 1_000_000:.2f}M"

    # If value is at least one thousand, show it in thousands.
    # Example: 52,000 becomes 52.0K.
    if abs(value) >= 1_000:
        return f"{value / 1_000:.1f}K"

    # For smaller numbers, show the full number.
    return f"{value:,.0f}"


def format_percent(value):
    """Convert a number into percentage text."""

    # If value is missing, return 0.0%.
    if pd.isna(value):
        return "0.0%"

    # Keep one decimal place.
    return f"{value:.1f}%"


def style_chart(fig, title, height=480):
    """Apply one consistent professional style to all charts."""

    # update_layout controls the overall appearance of the chart.
    fig.update_layout(

        # Chart title.
        title=title,

        # Use Plotly's clean white theme.
        template="plotly_white",

        # Chart height in pixels.
        height=height,

        # Move title slightly to the left, like many BI dashboards.
        title_x=0.02,

        # Set general font size and color.
        font=dict(size=13, color="#1f2937"),

        # Set chart backgrounds to white.
        paper_bgcolor="white",
        plot_bgcolor="white",

        # Add spacing around chart content.
        margin=dict(l=30, r=30, t=70, b=45),

        # Put legend horizontally above the chart.
        legend=dict(orientation="h", y=1.08),
    )

    # Remove vertical gridlines for a cleaner look.
    fig.update_xaxes(showgrid=False, zeroline=False)

    # Keep light horizontal gridlines to help read values.
    fig.update_yaxes(gridcolor="#e5e7eb", zeroline=False)

    # Return the formatted chart.
    return fig


def show_insight(title, bullets, conclusion):
    """Display a structured insight section after each chart."""

    # Convert a Python list of bullet points into Markdown bullet text.
    bullet_text = "\n".join([f"- {item}" for item in bullets])

    # Display a formatted insight box.
    display(Markdown(f"""
### 💡 {title}

{bullet_text}

**Conclusion:** {conclusion}
"""))


# 2. Load the Clean Master Data

This notebook does **not** read raw CSV files.

Instead, it reads the cleaned table created by the ETL pipeline:

```text
marketing.db → master_data
```

Why this matters:

- raw data may contain many tables
- the ETL pipeline already cleaned and joined them
- the notebook can focus on analysis, not data preparation


In [3]:
# ============================================================
# 3. LOAD CLEAN DATA FROM SQLITE
# ============================================================

# Get the folder where the notebook is currently running.
current_path = Path.cwd()

# If marketing.db is in the current folder, current folder is project root.
if (current_path / "marketing.db").exists():
    project_root = current_path

# If notebook is opened from notebooks/, marketing.db is one folder above.
elif (current_path.parent / "marketing.db").exists():
    project_root = current_path.parent

# If marketing.db is not found, stop and explain what to do.
else:
    raise FileNotFoundError("marketing.db not found. Run `python main.py` first.")

# Build the full path to the SQLite database.
db_path = project_root / "marketing.db"

# Create a database connection engine.
engine = create_engine(f"sqlite:///{db_path}")

# Read the master_data table from SQLite into a pandas dataframe.
df = pd.read_sql("SELECT * FROM master_data", engine)

# List of columns that should behave like dates.
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

# Convert available date columns from text to datetime.
for column in date_columns:

    # Only convert the column if it exists in the dataframe.
    if column in df.columns:

        # errors='coerce' means invalid dates become missing values instead of crashing.
        df[column] = pd.to_datetime(df[column], errors="coerce")

# Print number of rows and columns.
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

# Show first 5 rows to understand the dataset.
df.head()


Rows: 99,441
Columns: 36


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,payment_value,payment_installments,payment_count,primary_payment_type,payment_types,item_count,total_item_price,total_freight_value,unique_products,unique_sellers,main_product_id,main_seller_id,main_product_category,main_product_category_english,main_seller_state,main_seller_city,review_score,review_count,has_review_comment,customer_lat,customer_lng,delivery_delay_days,is_late_delivery,delivery_time_days
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,38.71,3.00,3.00,voucher,"credit_card, voucher",1.00,29.99,8.72,1.00,1.00,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,utilidades_domesticas,housewares,SP,maua,4.00,1.00,1.00,-23.58,-46.59,-8.00,0,8.00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,141.46,1.00,1.00,boleto,boleto,1.00,118.70,22.76,1.00,1.00,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,perfumaria,perfumery,SP,belo horizonte,4.00,1.00,1.00,-12.18,-44.66,-6.00,0,13.00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,179.12,3.00,1.00,credit_card,credit_card,1.00,159.90,19.22,1.00,1.00,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,automotivo,auto,SP,guariba,5.00,1.00,0.00,-16.75,-48.51,-18.00,0,9.00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,72.20,1.00,1.00,credit_card,credit_card,1.00,45.00,27.20,1.00,1.00,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,pet_shop,pet_shop,MG,belo horizonte,5.00,1.00,1.00,-5.77,-35.27,-13.00,0,13.00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,28.62,1.00,1.00,credit_card,credit_card,1.00,19.90,8.72,1.00,1.00,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,papelaria,stationery,SP,mogi das cruzes,5.00,1.00,0.00,-23.68,-46.51,-10.00,0,2.00


# 3. Data Quality Snapshot

## What is this chart for?

This section checks whether the dataset is safe for analysis.

## What is being calculated?

- total rows
- total columns
- unique orders
- duplicate order rows
- missing payment values
- columns with the most missing values

## Why this matters professionally

If `order_id` is duplicated in an order-level table, gross payment value calculations may be inflated.

If payment values are missing, gross payment value KPIs may be understated.


In [4]:
# ============================================================
# 4. DATA QUALITY SNAPSHOT
# ============================================================

# Count duplicate order_id rows.
# In an order-level table, this should ideally be 0.
duplicate_order_rows = df["order_id"].duplicated().sum()

# Count missing payment_value records.
# Missing payment values can affect gross payment value calculations.
missing_payment_values = df["payment_value"].isna().sum()

# Count missing values in every column.
missing_values = df.isna().sum().sort_values(ascending=False)

# Keep only columns where missing values exist.
# Show top 15 to keep chart readable.
missing_values_nonzero = missing_values[missing_values > 0].head(15)

# Create a small table with important quality metrics.
quality_summary = pd.DataFrame({
    "Metric": [
        "Total rows",
        "Total columns",
        "Unique orders",
        "Duplicate order_id rows",
        "Missing payment_value records"
    ],
    "Value": [
        len(df),
        df.shape[1],
        df["order_id"].nunique(),
        duplicate_order_rows,
        missing_payment_values
    ]
})

# Display quality summary table.
display(quality_summary)

# Only create missing-value chart if there are missing values.
if len(missing_values_nonzero) > 0:

    # Convert missing value series into a dataframe.
    missing_df = missing_values_nonzero.reset_index()

    # Rename columns for readability.
    missing_df.columns = ["column", "missing_values"]

    # Create horizontal bar chart.
    # Horizontal bars are easier to read when column names are long.
    fig = px.bar(
        missing_df.sort_values("missing_values"),
        x="missing_values",
        y="column",
        orientation="h",
        text="missing_values",
        color_discrete_sequence=["#2563EB"],
    )

    # Show missing values as labels outside bars.
    fig.update_traces(texttemplate="%{text:,}", textposition="outside")

    # Add clear axis labels.
    fig.update_layout(xaxis_title="Missing Values", yaxis_title="Column")

    # Apply common chart style.
    fig = style_chart(fig, "Top Columns with Missing Values", height=520)

    # Display chart.
    fig.show()

    # Store highest missing column for conclusion.
    top_missing_column = missing_df.iloc[0]["column"]
    top_missing_count = missing_df.iloc[0]["missing_values"]

else:

    # If there are no missing values, store safe defaults.
    top_missing_column = "None"
    top_missing_count = 0

# Display specific insight.
show_insight(
    "Data Quality Snapshot",
    [
        f"The dataset contains {len(df):,} rows and {df.shape[1]:,} columns.",
        f"Duplicate order_id rows: {duplicate_order_rows:,}.",
        f"Missing payment_value records: {missing_payment_values:,}.",
        f"Highest missing-value column among displayed fields: {top_missing_column} ({top_missing_count:,}).",
    ],
    "The most important result is duplicate order_id rows. If this is 0, the master dataset is safely modeled at one row per order."
)


,Metric,Value
0,Total rows,99441
1,Total columns,36
2,Unique orders,99441
3,Duplicate order_id rows,0
4,Missing payment_value records,0



### 💡 Data Quality Snapshot

- The dataset contains 99,441 rows and 36 columns.
- Duplicate order_id rows: 0.
- Missing payment_value records: 0.
- Highest missing-value column among displayed fields: delivery_time_days (2,965).

**Conclusion:** The most important result is duplicate order_id rows. If this is 0, the master dataset is safely modeled at one row per order.


# 4. Executive KPI Overview

## What is this section for?

This section calculates the main marketplace KPIs.

## KPI formulas

```text
Gross Payment Value = sum(payment_value)
Total Orders = count unique order_id
Average Gross Order Value = Gross Payment Value / Total Orders
Unique Customers = count unique customer_unique_id
Orders per Customer = Total Orders / Unique Customers
Repeat Customer Rate = Repeat Customers / Unique Customers
```

## Why this matters professionally

These KPIs are what business stakeholders usually look at first.


In [5]:
# ============================================================
# 5. EXECUTIVE KPI OVERVIEW
# ============================================================

# Gross Payment Value is the sum of payment_value.
# Important: this can include canceled orders if they have recorded payments.
gross_payment_value = df["payment_value"].sum()

# Total orders are calculated by counting unique order IDs.
# We use nunique() to avoid double counting.
total_orders = df["order_id"].nunique()

# Delivered Revenue counts only delivered orders.
# This is closer to recognized/real business revenue.
delivered_revenue = (
    df.loc[df["order_status"] == "delivered", "payment_value"].sum()
    if {"order_status", "payment_value"}.issubset(df.columns)
    else 0
)

# Canceled Gross Payment Value explains why canceled orders can still show payment value.
canceled_gross_payment_value = (
    df.loc[df["order_status"] == "canceled", "payment_value"].sum()
    if {"order_status", "payment_value"}.issubset(df.columns)
    else 0
)

# Count delivered, canceled, and unavailable orders.
delivered_orders = (
    df.loc[df["order_status"] == "delivered", "order_id"].nunique()
    if {"order_status", "order_id"}.issubset(df.columns)
    else 0
)

canceled_orders = (
    df.loc[df["order_status"] == "canceled", "order_id"].nunique()
    if {"order_status", "order_id"}.issubset(df.columns)
    else 0
)

unavailable_orders = (
    df.loc[df["order_status"] == "unavailable", "order_id"].nunique()
    if {"order_status", "order_id"}.issubset(df.columns)
    else 0
)

# Cancellation Rate = canceled orders / total orders.
cancellation_rate = canceled_orders / total_orders * 100 if total_orders > 0 else 0

# Delivery Success Rate = delivered orders / total orders.
delivery_success_rate = delivered_orders / total_orders * 100 if total_orders > 0 else 0

# Unique customers are calculated using customer_unique_id.
# This is important because customer_id can be order-specific in Olist.
unique_customers = df["customer_unique_id"].nunique()

# Average Gross Order Value shows recorded payment value per order.
average_order_value = gross_payment_value / total_orders if total_orders > 0 else 0

# Orders per customer shows average purchase frequency.
orders_per_customer = total_orders / unique_customers if unique_customers > 0 else 0

# Count how many orders each customer placed.
customer_order_counts = df.groupby("customer_unique_id")["order_id"].nunique()

# Customers with one order.
one_time_customers = (customer_order_counts == 1).sum()

# Customers with more than one order.
repeat_customers = (customer_order_counts > 1).sum()

# Repeat customer rate as a percentage.
repeat_customer_rate = repeat_customers / unique_customers * 100 if unique_customers > 0 else 0

# One-time customer rate as a percentage.
one_time_customer_rate = one_time_customers / unique_customers * 100 if unique_customers > 0 else 0

# Average review score helps connect business performance with customer satisfaction.
average_review_score = (
    df["review_score"].dropna().mean()
    if "review_score" in df.columns
    else 0
)

# Keep old variable name for compatibility with later cells in this notebook.
# From now on, remember: total_revenue means Gross Payment Value.
total_revenue = gross_payment_value

# Display KPIs in a readable table.
kpi_table = pd.DataFrame({
    "KPI": [
        "Gross Payment Value",
        "Delivered Revenue",
        "Canceled Gross Payment Value",
        "Total Orders",
        "Delivered Orders",
        "Canceled Orders",
        "Cancellation Rate",
        "Delivery Success Rate",
        "Average Gross Order Value",
        "Unique Customers",
        "Orders per Customer",
        "Repeat Customer Rate",
        "Average Review Score"
    ],
    "Value": [
        format_currency(gross_payment_value),
        format_currency(delivered_revenue),
        format_currency(canceled_gross_payment_value),
        f"{total_orders:,}",
        f"{delivered_orders:,}",
        f"{canceled_orders:,}",
        format_percent(cancellation_rate),
        format_percent(delivery_success_rate),
        format_currency(average_order_value),
        f"{unique_customers:,}",
        f"{orders_per_customer:.2f}",
        format_percent(repeat_customer_rate),
        f"{average_review_score:.2f}"
    ]
})

display(kpi_table)

show_insight(
    "Executive KPI Overview",
    [
        f"Gross Payment Value is {format_currency(gross_payment_value)}.",
        f"Delivered Revenue is {format_currency(delivered_revenue)}.",
        f"Canceled Gross Payment Value is {format_currency(canceled_gross_payment_value)}.",
        f"There are {total_orders:,} unique orders.",
        f"Canceled orders: {canceled_orders:,} ({cancellation_rate:.2f}%).",
        f"Delivery success rate is {delivery_success_rate:.2f}%.",
        f"Repeat customer rate is {repeat_customer_rate:.1f}%.",
    ],
    "Gross Payment Value and Delivered Revenue should be separated because canceled orders can still have recorded payment values."
)


,KPI,Value
0,Total Revenue,16.01M
1,Total Orders,"99,441"
2,Average Order Value,161
3,Unique Customers,"96,096"
4,Orders per Customer,1.03
5,One-Time Customer Rate,96.9%
6,Repeat Customer Rate,3.1%



### 💡 Executive KPI Overview

- Total revenue is 16.01M.
- The dataset contains 99,441 unique orders.
- Average order value is 161.
- There are 96,096 unique customers.
- Repeat customer rate is 3.1%.

**Conclusion:** The business has meaningful order volume, but the repeat customer rate of 3.1% suggests a customer retention opportunity.


# 5. Monthly Gross Payment Value and Order Volume


This chart combines **monthly gross payment value** and **monthly order volume**.

Chart design:

- bars = revenue
- line = number of orders
- second y-axis = orders, because orders and revenue are different scales

Business question:

> Did revenue move because order volume changed?


In [6]:
# ============================================================
# 6. MONTHLY REVENUE AND ORDER VOLUME
# ============================================================

# Remove rows without purchase date because they cannot be assigned to a month.
monthly_df = df.dropna(subset=["order_purchase_timestamp"]).copy()

# Create a month label from purchase timestamp.
# Example: 2017-08-15 becomes 2017-08.
monthly_df["order_month"] = monthly_df["order_purchase_timestamp"].dt.to_period("M").astype(str)

# Group by month and calculate monthly gross payment value and monthly orders.
monthly_summary = (
    monthly_df
    .groupby("order_month")
    .agg(
        revenue=("payment_value", "sum"),
        orders=("order_id", "nunique")
    )
    .reset_index()
)

# Calculate average order value per month.
monthly_summary["aov"] = monthly_summary["revenue"] / monthly_summary["orders"]

# Find the month with the highest revenue.
best_month_row = monthly_summary.loc[monthly_summary["revenue"].idxmax()]
best_month = best_month_row["order_month"]
best_month_revenue = best_month_row["revenue"]
best_month_orders = best_month_row["orders"]

# Create an empty figure because we want to add two chart types.
fig = go.Figure()

# Add revenue as bars.
fig.add_trace(
    go.Bar(
        x=monthly_summary["order_month"],
        y=monthly_summary["revenue"],
        name="Revenue",
        marker_color="#93C5FD",
        hovertemplate="Month: %{x}<br>Revenue: %{y:,.0f}<extra></extra>",
    )
)

# Add orders as a line.
# We use yaxis='y2' so order count appears on the right axis.
fig.add_trace(
    go.Scatter(
        x=monthly_summary["order_month"],
        y=monthly_summary["orders"],
        name="Orders",
        mode="lines+markers",
        line=dict(color="#111827", width=3),
        yaxis="y2",
        hovertemplate="Month: %{x}<br>Orders: %{y:,.0f}<extra></extra>",
    )
)

# Define left axis for revenue and right axis for orders.
fig.update_layout(
    xaxis_title="Month",
    yaxis=dict(title="Revenue"),
    yaxis2=dict(title="Orders", overlaying="y", side="right"),
)

# Apply professional styling and show chart.
fig = style_chart(fig, "Monthly Gross Payment Value and Order Volume", height=520)
fig.show()

show_insight(
    "Revenue Trend Insight",
    [
        f"The strongest revenue month was {best_month}.",
        f"{best_month} generated {format_currency(best_month_revenue)}.",
        f"{best_month} had {best_month_orders:,.0f} orders.",
    ],
    f"The peak month, {best_month}, should be investigated for seasonality, campaign activity, marketplace growth, or category mix."
)



### 💡 Revenue Trend Insight

- The strongest revenue month was 2017-11.
- 2017-11 generated 1.19M.
- 2017-11 had 7,544 orders.

**Conclusion:** The peak month, 2017-11, should be investigated for seasonality, campaign activity, marketplace growth, or category mix.


# 6. Monthly Gross Payment Value Growth


This chart shows month-over-month growth.

Calculation:

```text
Gross Payment Value Growth % = (current month revenue - previous month revenue) / previous month revenue
```

Business question:

> Is the marketplace accelerating or slowing down over time?


In [7]:
# ============================================================
# 7. MONTHLY REVENUE GROWTH
# ============================================================

# Copy monthly summary so we do not modify the original dataframe.
growth_df = monthly_summary.copy()

# pct_change() calculates month-over-month percentage change.
growth_df["revenue_growth_pct"] = growth_df["revenue"].pct_change() * 100

# Remove the first month because it has no previous month for comparison.
growth_plot_df = growth_df.dropna(subset=["revenue_growth_pct"]).copy()

# Find the month with highest gross payment value growth.
best_growth_row = growth_plot_df.loc[growth_plot_df["revenue_growth_pct"].idxmax()]

# Find the month with lowest gross payment value growth.
worst_growth_row = growth_plot_df.loc[growth_plot_df["revenue_growth_pct"].idxmin()]

# Create bar chart for growth percentage.
fig = px.bar(
    growth_plot_df,
    x="order_month",
    y="revenue_growth_pct",
    text="revenue_growth_pct",
    color_discrete_sequence=["#2563EB"],
)

# Format text labels as percentages.
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")

# Add axis labels.
fig.update_layout(xaxis_title="Month", yaxis_title="Gross Payment Value Growth (%)")

# Apply style and display.
fig = style_chart(fig, "Month-over-Month Gross Payment Value Growth", height=520)
fig.show()

show_insight(
    "Gross Payment Value Growth Insight",
    [
        f"Highest monthly growth was {best_growth_row['revenue_growth_pct']:.1f}% in {best_growth_row['order_month']}.",
        f"Lowest monthly growth was {worst_growth_row['revenue_growth_pct']:.1f}% in {worst_growth_row['order_month']}.",
    ],
    "Growth analysis helps identify acceleration and slowdown periods, which should be connected back to campaigns, seasonality, and operational changes."
)



### 💡 Revenue Growth Insight

- Highest monthly growth was 705751.4% in 2017-01.
- Lowest monthly growth was -100.0% in 2016-12.

**Conclusion:** Growth analysis helps identify acceleration and slowdown periods, which should be connected back to campaigns, seasonality, and operational changes.


# 7. Top Product Categories by Gross Payment Value


This chart ranks categories by revenue.

Chart design:

- horizontal bar chart because category names can be long
- revenue labels shown outside bars
- top 12 categories only to keep chart readable


In [8]:
# ============================================================
# 8. TOP PRODUCT CATEGORIES BY REVENUE
# ============================================================

# Use English product category if it exists; otherwise use original category.
category_column = "main_product_category_english" if "main_product_category_english" in df.columns else "main_product_category"

# Group by category and calculate revenue and order count.
category_revenue = (
    df.dropna(subset=[category_column])
    .groupby(category_column)
    .agg(
        revenue=("payment_value", "sum"),
        orders=("order_id", "nunique")
    )
    .sort_values("revenue", ascending=False)
)

# Store top category values for conclusions.
top_category = category_revenue.index[0]
top_category_revenue = category_revenue.iloc[0]["revenue"]

# Calculate how much revenue top categories contribute.
top_5_category_share = category_revenue.head(5)["revenue"].sum() / total_revenue * 100
top_10_category_share = category_revenue.head(10)["revenue"].sum() / total_revenue * 100

# Select top 12 categories for plotting.
top_categories = category_revenue.head(12).reset_index()

# Create horizontal bar chart.
fig = px.bar(
    top_categories.sort_values("revenue"),
    x="revenue",
    y=category_column,
    orientation="h",
    text="revenue",
    hover_data=["orders"],
    color_discrete_sequence=["#2563EB"],
)

# Format value labels.
fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")

# Add axis titles.
fig.update_layout(xaxis_title="Revenue", yaxis_title="Product Category")

# Apply style and display.
fig = style_chart(fig, "Top Product Categories by Gross Payment Value", height=620)
fig.show()

show_insight(
    "Category Gross Payment Value Insight",
    [
        f"The top revenue category is {top_category}.",
        f"{top_category} generated {format_currency(top_category_revenue)}.",
        f"Top 5 categories contributed {top_5_category_share:.1f}% of gross payment value.",
        f"Top 10 categories contributed {top_10_category_share:.1f}% of gross payment value.",
    ],
    "Revenue is concentrated in a limited number of categories. These categories should receive priority in marketing, inventory, and seller partnership planning."
)



### 💡 Category Revenue Insight

- The top revenue category is health_beauty.
- health_beauty generated 1.44M.
- Top 5 categories contributed 38.8% of total revenue.
- Top 10 categories contributed 61.8% of total revenue.

**Conclusion:** Revenue is concentrated in a limited number of categories. These categories should receive priority in marketing, inventory, and seller partnership planning.


# 8. Category Pareto Analysis


A Pareto chart combines bars and a cumulative percentage line.

Calculation:

```text
category gross payment value share = category gross payment value / total category gross payment value
cumulative share = running total of revenue share
```

Business question:

> How many categories explain most revenue?


In [9]:
# ============================================================
# 9. CATEGORY PARETO ANALYSIS
# ============================================================

# Convert category gross payment value index into a normal dataframe.
pareto_df = category_revenue.reset_index().copy()

# Calculate each category's share of revenue.
pareto_df["revenue_share"] = pareto_df["revenue"] / pareto_df["revenue"].sum() * 100

# Calculate cumulative revenue share from highest to lowest category.
pareto_df["cumulative_share"] = pareto_df["revenue_share"].cumsum()

# Keep top 20 categories for readable chart.
pareto_top = pareto_df.head(20)

# Calculate how many categories are needed to reach about 80% revenue.
categories_to_80 = (pareto_df["cumulative_share"] <= 80).sum() + 1

# Create figure manually because we combine bar and line.
fig = go.Figure()

# Add revenue bars.
fig.add_trace(
    go.Bar(
        x=pareto_top[category_column],
        y=pareto_top["revenue"],
        name="Revenue",
        marker_color="#60A5FA",
    )
)

# Add cumulative share line on second y-axis.
fig.add_trace(
    go.Scatter(
        x=pareto_top[category_column],
        y=pareto_top["cumulative_share"],
        name="Cumulative Revenue Share",
        mode="lines+markers",
        yaxis="y2",
        line=dict(color="#111827", width=3),
    )
)

# Configure left and right axes.
fig.update_layout(
    xaxis_title="Product Category",
    yaxis=dict(title="Revenue"),
    yaxis2=dict(title="Cumulative Share (%)", overlaying="y", side="right", range=[0, 100]),
)

# Rotate category labels because they are long.
fig.update_xaxes(tickangle=45)

# Apply style and display.
fig = style_chart(fig, "Category Pareto Analysis", height=650)
fig.show()

show_insight(
    "Category Pareto Insight",
    [
        f"Top 5 categories explain {top_5_category_share:.1f}% of revenue.",
        f"Top 10 categories explain {top_10_category_share:.1f}% of revenue.",
        f"About {categories_to_80} categories are needed to reach roughly 80% of revenue.",
    ],
    "This chart shows whether revenue is concentrated or diversified across categories. High concentration can help focus strategy, but also creates dependency risk."
)



### 💡 Category Pareto Insight

- Top 5 categories explain 38.8% of revenue.
- Top 10 categories explain 61.8% of revenue.
- About 17 categories are needed to reach roughly 80% of revenue.

**Conclusion:** This chart shows whether revenue is concentrated or diversified across categories. High concentration can help focus strategy, but also creates dependency risk.


# 9. Category Treemap


A treemap shows category gross payment value as rectangles.

Chart design:

- bigger box = more revenue
- color intensity also follows revenue
- useful for visually spotting dominant categories


In [10]:
# ============================================================
# 10. CATEGORY TREEMAP
# ============================================================

# Select top 20 categories so the treemap remains readable.
treemap_df = category_revenue.head(20).reset_index()

# Create treemap.
# path defines the hierarchy; here we only have one level: category.
# values controls the size of each box.
fig = px.treemap(
    treemap_df,
    path=[category_column],
    values="revenue",
    color="revenue",
    color_continuous_scale="Blues",
    hover_data=["orders"],
)

# Apply chart styling.
fig = style_chart(fig, "Category Gross Payment Value Treemap", height=600)

# Display chart.
fig.show()

show_insight(
    "Treemap Insight",
    [
        f"The largest block is {top_category}.",
        f"{top_category} generated {format_currency(top_category_revenue)}.",
        "The visual size of each block represents revenue contribution.",
    ],
    "The treemap quickly shows which categories dominate marketplace revenue and which categories are smaller contributors."
)



### 💡 Treemap Insight

- The largest block is health_beauty.
- health_beauty generated 1.44M.
- The visual size of each block represents revenue contribution.

**Conclusion:** The treemap quickly shows which categories dominate marketplace revenue and which categories are smaller contributors.


# 10. Customer Retention: One-Time vs Repeat Customers


This chart compares one-time buyers with repeat buyers.

Calculation:

```text
one-time customer = customer with exactly 1 order
repeat customer = customer with more than 1 order
```


In [11]:
# ============================================================
# 11. CUSTOMER RETENTION ANALYSIS
# ============================================================

# Create a small dataframe for customer types.
customer_types = pd.DataFrame({
    "customer_type": ["One-time customers", "Repeat customers"],
    "customers": [one_time_customers, repeat_customers]
})

# Calculate share of total customers.
customer_types["share"] = customer_types["customers"] / unique_customers * 100

# Create bar chart.
fig = px.bar(
    customer_types,
    x="customer_type",
    y="customers",
    text="customers",
    color="customer_type",
    color_discrete_sequence=["#93C5FD", "#2563EB"],
)

# Format labels and remove legend because x-axis already explains categories.
fig.update_traces(texttemplate="%{text:,}", textposition="outside")
fig.update_layout(xaxis_title="Customer Type", yaxis_title="Number of Customers", showlegend=False)

# Apply style and display.
fig = style_chart(fig, "One-Time vs Repeat Customers", height=480)
fig.show()

show_insight(
    "Customer Retention Insight",
    [
        f"One-time customers: {one_time_customers:,} ({one_time_customer_rate:.1f}%).",
        f"Repeat customers: {repeat_customers:,} ({repeat_customer_rate:.1f}%).",
        f"Average orders per customer: {orders_per_customer:.2f}.",
    ],
    f"The marketplace has a retention opportunity because {one_time_customer_rate:.1f}% of customers purchased only once."
)



### 💡 Customer Retention Insight

- One-time customers: 93,099 (96.9%).
- Repeat customers: 2,997 (3.1%).
- Average orders per customer: 1.03.

**Conclusion:** The marketplace has a retention opportunity because 96.9% of customers purchased only once.


# 11. Customer Cohort Retention Heatmap


Cohort analysis groups customers by their first purchase month.

Calculation logic:

```text
cohort month = customer's first purchase month
month 0 = first purchase month
month 1 = one month after first purchase
retention % = active customers in later month / customers in month 0
```

This chart is useful for understanding retention over time.


In [12]:
# ============================================================
# 12. CUSTOMER COHORT RETENTION HEATMAP
# ============================================================

# Keep only columns needed for cohort analysis.
cohort_df = df[["customer_unique_id", "order_id", "order_purchase_timestamp"]].dropna().copy()

# Convert purchase date into month period.
cohort_df["order_month"] = cohort_df["order_purchase_timestamp"].dt.to_period("M")

# Find the first purchase month for each customer.
cohort_df["cohort_month"] = cohort_df.groupby("customer_unique_id")["order_month"].transform("min")

# Calculate how many months passed since first purchase.
cohort_df["months_since_first_purchase"] = (
    (cohort_df["order_month"].dt.year - cohort_df["cohort_month"].dt.year) * 12
    + (cohort_df["order_month"].dt.month - cohort_df["cohort_month"].dt.month)
)

# Count unique customers for each cohort and month number.
cohort_counts = (
    cohort_df
    .groupby(["cohort_month", "months_since_first_purchase"])["customer_unique_id"]
    .nunique()
    .reset_index()
)

# Create matrix for heatmap.
# Rows are cohort months; columns are months since first purchase.
cohort_pivot = cohort_counts.pivot(
    index="cohort_month",
    columns="months_since_first_purchase",
    values="customer_unique_id"
)

# Convert counts into retention percentage.
cohort_retention = cohort_pivot.divide(cohort_pivot.iloc[:, 0], axis=0) * 100

# Limit to first 12 months for readability.
cohort_retention = cohort_retention.iloc[:, :13]

# Plotly cannot display pandas Period values directly.
# Convert index and columns to strings.
cohort_retention.index = cohort_retention.index.astype(str)
cohort_retention.columns = cohort_retention.columns.astype(str)

# Calculate average month-1 retention if month 1 exists.
if "1" in cohort_retention.columns:
    avg_month_1_retention = cohort_retention["1"].mean()
else:
    avg_month_1_retention = np.nan

# Create heatmap.
fig = px.imshow(
    cohort_retention,
    text_auto=".1f",
    aspect="auto",
    color_continuous_scale="Blues",
    labels={
        "x": "Months Since First Purchase",
        "y": "Customer Cohort",
        "color": "Retention %"
    }
)

# Label axes and colorbar.
fig.update_layout(
    xaxis_title="Months Since First Purchase",
    yaxis_title="Customer Cohort",
    coloraxis_colorbar_title="Retention %"
)

# Apply style and display.
fig = style_chart(fig, "Customer Cohort Retention Heatmap", height=650)
fig.show()

show_insight(
    "Cohort Retention Insight",
    [
        "Month 0 is always 100% because it is the first purchase month.",
        f"Average month-1 retention is {avg_month_1_retention:.1f}%." if not pd.isna(avg_month_1_retention) else "Month-1 retention could not be calculated.",
        "Low retention after month 0 means many customers do not return.",
    ],
    "Cohort analysis confirms whether retention is a structural issue over time, not just a one-time KPI."
)



### 💡 Cohort Retention Insight

- Month 0 is always 100% because it is the first purchase month.
- Average month-1 retention is 5.2%.
- Low retention after month 0 means many customers do not return.

**Conclusion:** Cohort analysis confirms whether retention is a structural issue over time, not just a one-time KPI.


# 12. Gross Payment Value by Customer State


This chart shows geographic revenue concentration.

Chart design:

- bar chart
- top 15 states only
- hover shows orders and customer counts


In [13]:
# ============================================================
# 13. REVENUE BY CUSTOMER STATE
# ============================================================

# Group by customer state and calculate revenue, orders, and customers.
state_revenue = (
    df.dropna(subset=["customer_state"])
    .groupby("customer_state")
    .agg(
        revenue=("payment_value", "sum"),
        orders=("order_id", "nunique"),
        customers=("customer_unique_id", "nunique")
    )
    .sort_values("revenue", ascending=False)
    .reset_index()
)

# Store top state values for conclusion.
top_state = state_revenue.iloc[0]["customer_state"]
top_state_revenue = state_revenue.iloc[0]["revenue"]
top_state_share = top_state_revenue / total_revenue * 100

# Create bar chart for top 15 states.
fig = px.bar(
    state_revenue.head(15),
    x="customer_state",
    y="revenue",
    text="revenue",
    hover_data=["orders", "customers"],
    color_discrete_sequence=["#2563EB"],
)

# Format bar labels.
fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig.update_layout(xaxis_title="Customer State", yaxis_title="Revenue")

# Apply style and display.
fig = style_chart(fig, "Top Customer States by Revenue", height=520)
fig.show()

show_insight(
    "Regional Revenue Insight",
    [
        f"The highest revenue state is {top_state}.",
        f"{top_state} generated {format_currency(top_state_revenue)}.",
        f"{top_state} represents {top_state_share:.1f}% of gross payment value.",
    ],
    f"{top_state} is the strongest customer market. Regional marketing and logistics decisions should consider this concentration."
)



### 💡 Regional Revenue Insight

- The highest revenue state is SP.
- SP generated 6.00M.
- SP represents 37.5% of total revenue.

**Conclusion:** SP is the strongest customer market. Regional marketing and logistics decisions should consider this concentration.


# 13. Payment Method Share


This chart shows which payment method customers use most.

Calculation:

```text
payment method share = orders using method / total orders
```


In [14]:
# ============================================================
# 14. PAYMENT METHOD SHARE
# ============================================================

# Use primary_payment_type if ETL created it; otherwise use payment_type.
payment_column = "primary_payment_type" if "primary_payment_type" in df.columns else "payment_type"

# Count payment method share as percentage.
payment_share = (
    df[payment_column]
    .dropna()
    .value_counts(normalize=True)
    .mul(100)
    .reset_index()
)

# Rename columns for readability.
payment_share.columns = ["payment_type", "share"]

# Store top method for conclusion.
top_payment_method = payment_share.iloc[0]["payment_type"]
top_payment_share = payment_share.iloc[0]["share"]

# Create bar chart.
fig = px.bar(
    payment_share,
    x="payment_type",
    y="share",
    text="share",
    color_discrete_sequence=["#2563EB"],
)

fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(xaxis_title="Payment Method", yaxis_title="Share of Orders (%)")
fig = style_chart(fig, "Primary Payment Method Share", height=480)
fig.show()

show_insight(
    "Payment Method Insight",
    [
        f"The most used payment method is {top_payment_method}.",
        f"{top_payment_method} represents {top_payment_share:.1f}% of orders.",
    ],
    "The dominant payment method should be reliable because checkout issues there could affect many orders."
)



### 💡 Payment Method Insight

- The most used payment method is credit_card.
- credit_card represents 76.6% of orders.

**Conclusion:** The dominant payment method should be reliable because checkout issues there could affect many orders.


# 14. Order Status Distribution


This chart checks order fulfillment status.

Calculation:

```text
status share = orders with status / total orders
```


In [15]:
# ============================================================
# 15. ORDER STATUS DISTRIBUTION
# ============================================================

# Calculate percentage share of each order status.
order_status_share = (
    df["order_status"]
    .value_counts(normalize=True)
    .mul(100)
    .reset_index()
)

# Rename columns.
order_status_share.columns = ["order_status", "share"]

# Store most common status.
top_status = order_status_share.iloc[0]["order_status"]
top_status_share = order_status_share.iloc[0]["share"]

# Create bar chart.
fig = px.bar(
    order_status_share,
    x="order_status",
    y="share",
    text="share",
    color_discrete_sequence=["#2563EB"],
)

fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(xaxis_title="Order Status", yaxis_title="Share of Orders (%)")
fig = style_chart(fig, "Order Status Distribution", height=480)
fig.show()

show_insight(
    "Order Status Insight",
    [
        f"The most common status is {top_status}.",
        f"{top_status} represents {top_status_share:.1f}% of orders.",
    ],
    "A high delivered share indicates healthy fulfillment. Cancellation or unavailable shares should be monitored for operational issues."
)



### 💡 Order Status Insight

- The most common status is delivered.
- delivered represents 97.0% of orders.

**Conclusion:** A high delivered share indicates healthy fulfillment. Cancellation or unavailable shares should be monitored for operational issues.


# Operational KPI Summary

## What is this section for?

This section explains order fulfillment health using operational KPIs.

## What is being calculated?

```text
Canceled Orders = count unique order_id where order_status = canceled
Cancellation Rate = Canceled Orders / Total Orders
Delivered Orders = count unique order_id where order_status = delivered
Delivery Success Rate = Delivered Orders / Total Orders
Canceled Gross Payment Value = payment_value from canceled orders
```

## Why this matters professionally

A dashboard should not only show growth. It should also show problems.

Canceled orders, unavailable orders, and late delivery can explain why gross payment value does not fully convert into delivered revenue.


In [ ]:
# ============================================================
# 15B. OPERATIONAL KPI SUMMARY
# ============================================================

# Create a small operations table.
# This table helps explain fulfillment and cancellation health.
operations_kpis = pd.DataFrame({
    "Operational KPI": [
        "Delivered Orders",
        "Canceled Orders",
        "Unavailable Orders",
        "Cancellation Rate",
        "Delivery Success Rate",
        "Canceled Gross Payment Value",
        "Average Review Score"
    ],
    "Value": [
        f"{delivered_orders:,}",
        f"{canceled_orders:,}",
        f"{unavailable_orders:,}",
        format_percent(cancellation_rate),
        format_percent(delivery_success_rate),
        format_currency(canceled_gross_payment_value),
        f"{average_review_score:.2f}"
    ]
})

display(operations_kpis)

show_insight(
    "Operational Health Insight",
    [
        f"Delivered orders: {delivered_orders:,}.",
        f"Canceled orders: {canceled_orders:,}.",
        f"Unavailable orders: {unavailable_orders:,}.",
        f"Cancellation rate: {cancellation_rate:.2f}%.",
        f"Delivery success rate: {delivery_success_rate:.2f}%.",
        f"Canceled Gross Payment Value: {format_currency(canceled_gross_payment_value)}.",
    ],
    "Operational KPIs help explain whether recorded payment value is converting into successfully delivered orders."
)


# 15. Delivery Delay Distribution


This chart analyzes delivery delay.

Interpretation:

```text
negative delay = early delivery
zero = on estimated date
positive delay = late delivery
```


In [16]:
# ============================================================
# 16. DELIVERY DELAY DISTRIBUTION
# ============================================================

# Keep rows with delivery delay values.
delivery_df = df.dropna(subset=["delivery_delay_days"]).copy()

# Late delivery means delivery_delay_days is greater than 0.
late_delivery_rate = (delivery_df["delivery_delay_days"] > 0).mean() * 100 if len(delivery_df) > 0 else 0

# Average delay compared with estimated date.
avg_delivery_delay = delivery_df["delivery_delay_days"].mean() if len(delivery_df) > 0 else 0

# Average delivery time from purchase to customer delivery.
avg_delivery_time = df["delivery_time_days"].dropna().mean() if "delivery_time_days" in df.columns else np.nan

# Create histogram.
fig = px.histogram(
    delivery_df,
    x="delivery_delay_days",
    nbins=70,
    color_discrete_sequence=["#2563EB"],
)

# Add a vertical line at 0.
# This separates early/on-time deliveries from late deliveries.
fig.add_vline(
    x=0,
    line_dash="dash",
    line_color="#EF4444",
    annotation_text="Estimated delivery date",
)

fig.update_layout(xaxis_title="Delivery Delay Days", yaxis_title="Number of Orders")
fig = style_chart(fig, "Delivery Delay Distribution", height=520)
fig.show()

show_insight(
    "Delivery Performance Insight",
    [
        f"Late delivery rate: {late_delivery_rate:.1f}%.",
        f"Average delivery delay vs estimate: {avg_delivery_delay:.1f} days.",
        f"Average delivery time from purchase to delivery: {avg_delivery_time:.1f} days.",
    ],
    "Delivery performance is an operational risk. Late deliveries should be monitored by seller, state, and category."
)



### 💡 Delivery Performance Insight

- Late delivery rate: 6.8%.
- Average delivery delay vs estimate: -11.9 days.
- Average delivery time from purchase to delivery: 12.1 days.

**Conclusion:** Delivery performance is an operational risk. Late deliveries should be monitored by seller, state, and category.


# 16. Review Score vs Delivery Delay


This scatter plot connects logistics with customer satisfaction.

Calculation:

- x-axis = delivery delay days
- y-axis = review score
- correlation = relationship between delay and rating


In [17]:
# ============================================================
# 17. REVIEW SCORE VS DELIVERY DELAY
# ============================================================

# Only run this chart if both columns exist.
if "review_score" in df.columns and "delivery_delay_days" in df.columns:

    # Keep rows with both review score and delivery delay.
    review_delivery = df.dropna(subset=["review_score", "delivery_delay_days"]).copy()

    # Use a sample if there are many rows to keep chart readable.
    sample_size = min(5000, len(review_delivery))
    review_delivery_sample = review_delivery.sample(sample_size, random_state=42)

    # Calculate correlation between delay and review score.
    correlation = review_delivery["review_score"].corr(review_delivery["delivery_delay_days"])

    # Create scatter plot.
    fig = px.scatter(
        review_delivery_sample,
        x="delivery_delay_days",
        y="review_score",
        opacity=0.35,
        color_discrete_sequence=["#2563EB"],
    )

    fig.update_layout(xaxis_title="Delivery Delay Days", yaxis_title="Review Score")
    fig = style_chart(fig, "Review Score vs Delivery Delay", height=520)
    fig.show()

    show_insight(
        "Customer Experience Insight",
        [
            f"The correlation between delivery delay and review score is {correlation:.2f}.",
            "Positive delivery delay means late delivery.",
            "Lower review scores at high delays may indicate customer dissatisfaction.",
        ],
        "Delivery performance should be monitored together with review scores because logistics can influence customer experience."
    )

else:
    display(Markdown("Review score or delivery delay column is missing, so this chart was skipped."))



### 💡 Customer Experience Insight

- The correlation between delivery delay and review score is -0.27.
- Positive delivery delay means late delivery.
- Lower review scores at high delays may indicate customer dissatisfaction.

**Conclusion:** Delivery performance should be monitored together with review scores because logistics can influence customer experience.


# 17. Top Sellers by Gross Payment Value


This chart ranks sellers by revenue.

Business question:

> Which sellers are most important for marketplace revenue?


In [18]:
# ============================================================
# 18. TOP SELLERS BY REVENUE
# ============================================================

# Use main_seller_id if available; otherwise use seller_id.
seller_column = "main_seller_id" if "main_seller_id" in df.columns else "seller_id"

# Group by seller and calculate revenue and orders.
seller_revenue = (
    df.dropna(subset=[seller_column])
    .groupby(seller_column)
    .agg(
        revenue=("payment_value", "sum"),
        orders=("order_id", "nunique")
    )
    .sort_values("revenue", ascending=False)
)

# Store top seller metrics.
top_seller = seller_revenue.index[0]
top_seller_revenue = seller_revenue.iloc[0]["revenue"]
top_10_seller_share = seller_revenue.head(10)["revenue"].sum() / total_revenue * 100

# Select top 12 sellers for chart.
top_sellers = seller_revenue.head(12).reset_index()

# Create horizontal bar chart.
fig = px.bar(
    top_sellers.sort_values("revenue"),
    x="revenue",
    y=seller_column,
    orientation="h",
    text="revenue",
    hover_data=["orders"],
    color_discrete_sequence=["#2563EB"],
)

fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig.update_layout(xaxis_title="Revenue", yaxis_title="Seller ID")
fig = style_chart(fig, "Top Sellers by Gross Payment Value", height=620)
fig.show()

show_insight(
    "Seller Gross Payment Value Insight",
    [
        f"The top seller is {top_seller}.",
        f"The top seller generated {format_currency(top_seller_revenue)}.",
        f"The top 10 sellers contributed {top_10_seller_share:.1f}% of gross payment value.",
    ],
    "Top sellers are strategic partners. High concentration can create dependency risk if a few sellers generate a large share of revenue."
)



### 💡 Seller Revenue Insight

- The top seller is 4869f7a5dfa277a7dca6462dcf3b52b2.
- The top seller generated 253.1K.
- The top 10 sellers contributed 12.7% of total revenue.

**Conclusion:** Top sellers are strategic partners. High concentration can create dependency risk if a few sellers generate a large share of revenue.


# 18. Seller Pareto Analysis


This chart checks whether revenue is concentrated among a few sellers.

It uses the same idea as category Pareto analysis.


In [19]:
# ============================================================
# 19. SELLER PARETO ANALYSIS
# ============================================================

# Convert seller gross payment value into dataframe.
seller_pareto = seller_revenue.reset_index().copy()

# Calculate each seller's revenue share.
seller_pareto["revenue_share"] = seller_pareto["revenue"] / seller_pareto["revenue"].sum() * 100

# Calculate cumulative revenue share.
seller_pareto["cumulative_share"] = seller_pareto["revenue_share"].cumsum()

# Keep top 20 sellers for readability.
seller_pareto_top = seller_pareto.head(20)

# Calculate number of sellers needed to reach about 80% revenue.
sellers_to_80 = (seller_pareto["cumulative_share"] <= 80).sum() + 1

# Build combined bar + line chart.
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=seller_pareto_top[seller_column],
        y=seller_pareto_top["revenue"],
        name="Revenue",
        marker_color="#60A5FA",
    )
)

fig.add_trace(
    go.Scatter(
        x=seller_pareto_top[seller_column],
        y=seller_pareto_top["cumulative_share"],
        name="Cumulative Share",
        mode="lines+markers",
        yaxis="y2",
        line=dict(color="#111827", width=3),
    )
)

fig.update_layout(
    xaxis_title="Seller ID",
    yaxis=dict(title="Revenue"),
    yaxis2=dict(title="Cumulative Share (%)", overlaying="y", side="right", range=[0, 100]),
)

fig.update_xaxes(tickangle=45)
fig = style_chart(fig, "Seller Pareto Analysis", height=650)
fig.show()

show_insight(
    "Seller Pareto Insight",
    [
        f"Top 10 sellers generated {top_10_seller_share:.1f}% of revenue.",
        f"About {sellers_to_80} sellers are needed to reach roughly 80% of seller gross payment value.",
    ],
    "Seller Pareto analysis helps understand whether the marketplace depends on a small group of high-performing sellers."
)



### 💡 Seller Pareto Insight

- Top 10 sellers generated 12.7% of revenue.
- About 563 sellers are needed to reach roughly 80% of seller revenue.

**Conclusion:** Seller Pareto analysis helps understand whether the marketplace depends on a small group of high-performing sellers.


# 19. Payment Value Distribution


This box plot shows spread and outliers in payment values.

Why box plot?

- median is the middle line
- box shows typical range
- points outside are outliers


In [20]:
# ============================================================
# 20. PAYMENT VALUE DISTRIBUTION
# ============================================================

# Calculate distribution metrics.
median_payment = df["payment_value"].median()
p95_payment = df["payment_value"].quantile(0.95)
max_payment = df["payment_value"].max()

# Create box plot.
fig = px.box(
    df,
    y="payment_value",
    points="outliers",
    color_discrete_sequence=["#2563EB"],
)

fig.update_layout(yaxis_title="Payment Value")
fig = style_chart(fig, "Payment Value Distribution", height=520)
fig.show()

show_insight(
    "Payment Distribution Insight",
    [
        f"Median payment value is {format_currency(median_payment)}.",
        f"95th percentile payment value is {format_currency(p95_payment)}.",
        f"Maximum payment value is {format_currency(max_payment)}.",
    ],
    "Payment values are usually skewed in e-commerce. Very large orders can pull the average upward, so median and percentile values should also be considered."
)



### 💡 Payment Distribution Insight

- Median payment value is 105.
- 95th percentile payment value is 453.
- Maximum payment value is 13.7K.

**Conclusion:** Payment values are usually skewed in e-commerce. Very large orders can pull the average upward, so median and percentile values should also be considered.


# 20. Final Data-Backed Conclusions

This final section summarizes the analysis using actual numbers calculated above.

A professional conclusion should include:

- exact values
- percentages
- business interpretation
- recommended area of attention


In [21]:
# ============================================================
# 21. FINAL DATA-BACKED CONCLUSIONS
# ============================================================

# Display a final markdown summary using the variables calculated earlier.
display(Markdown(f"""
# ✅ Final EDA Conclusions

## 1. Gross Payment Value Performance

The marketplace recorded **{format_currency(gross_payment_value)}** in Gross Payment Value across **{total_orders:,} unique orders**.

Delivered Revenue was **{format_currency(delivered_revenue)}**.

The Average Gross Order Value was **{format_currency(average_order_value)}**.

The strongest gross payment value month was **{best_month}**, with **{format_currency(best_month_revenue)}** from **{best_month_orders:,.0f} orders**.

**Conclusion:**  
Gross Payment Value shows recorded payment activity, but Delivered Revenue is the cleaner business revenue metric because it only counts delivered orders. The best-performing month, **{best_month}**, should be studied further for seasonality, growth, campaigns, or category mix.

---

## 2. Cancellation and Fulfillment Health

The dataset contains **{canceled_orders:,} canceled orders**.

The Cancellation Rate was **{cancellation_rate:.2f}%**.

Canceled Gross Payment Value was **{format_currency(canceled_gross_payment_value)}**.

Delivered Orders were **{delivered_orders:,}**, giving a Delivery Success Rate of **{delivery_success_rate:.2f}%**.

Unavailable Orders were **{unavailable_orders:,}**.

**Conclusion:**  
Canceled orders can still have recorded payment value, so it is important to separate Gross Payment Value from Delivered Revenue. Cancellation Rate and Delivery Success Rate are important operational KPIs for understanding marketplace health.

---

## 3. Category Concentration

The highest gross payment value category was **{top_category}**, generating **{format_currency(top_category_revenue)}**.

The top 5 categories generated **{top_5_category_share:.1f}%** of total Gross Payment Value.  
The top 10 categories generated **{top_10_category_share:.1f}%** of total Gross Payment Value.

About **{categories_to_80} categories** are needed to reach roughly 80% of category Gross Payment Value.

**Conclusion:**  
Gross Payment Value is concentrated in a limited set of product categories. The business should prioritize these categories for marketing, inventory planning, and seller partnerships, while also managing concentration risk.

---

## 4. Customer Retention

The dataset contains **{unique_customers:,} unique customers**.

- **{one_time_customers:,} customers** purchased only once.
- **{repeat_customers:,} customers** purchased more than once.
- The one-time customer rate is **{one_time_customer_rate:.1f}%**.
- The repeat customer rate is **{repeat_customer_rate:.1f}%**.
- Average orders per customer is **{orders_per_customer:.2f}**.

**Conclusion:**  
The marketplace has a clear retention opportunity. Since **{one_time_customer_rate:.1f}%** of customers purchased only once, campaigns such as remarketing, email follow-ups, loyalty offers, and personalized recommendations could improve repeat purchasing.

---

## 5. Regional Gross Payment Value

The strongest customer state was **{top_state}**, generating **{format_currency(top_state_revenue)}**, or **{top_state_share:.1f}%** of total Gross Payment Value.

**Conclusion:**  
Gross Payment Value is geographically concentrated. The business should understand what makes **{top_state}** strong and use those learnings for other regions.

---

## 6. Payment Behavior

The most used payment method was **{top_payment_method}**, representing **{top_payment_share:.1f}%** of orders.

**Conclusion:**  
The dominant payment method should be reliable because checkout issues in this method could affect a large share of orders.

---

## 7. Delivery Performance

The late delivery rate was **{late_delivery_rate:.1f}%**.

Average delivery delay was **{avg_delivery_delay:.1f} days** compared with the estimated delivery date.

Average delivery time was **{avg_delivery_time:.1f} days** from purchase to customer delivery.

Average review score was **{average_review_score:.2f}**.

**Conclusion:**  
Delivery performance is a key operational area. Late delivery can damage customer satisfaction and should be monitored by seller, customer state, and product category.

---

## 8. Seller Concentration

The top seller was **{top_seller}**, generating **{format_currency(top_seller_revenue)}**.

The top 10 sellers contributed **{top_10_seller_share:.1f}%** of total Gross Payment Value.

About **{sellers_to_80} sellers** are needed to reach roughly 80% of seller Gross Payment Value.

**Conclusion:**  
High-performing sellers are important strategic partners. If gross payment value depends heavily on a small number of sellers, the marketplace should manage dependency risk and support seller quality.

---

## 9. Payment Value Distribution

The median payment value was **{format_currency(median_payment)}**.

The 95th percentile payment value was **{format_currency(p95_payment)}**.

The maximum payment value was **{format_currency(max_payment)}**.

**Conclusion:**  
Payment values are skewed, so Average Gross Order Value should not be interpreted alone. Median and percentile values provide a better view of typical customer spend.

---

## 10. Data Quality and Data Grain

The final dataset contains **{len(df):,} rows** and **{total_orders:,} unique orders**.

Duplicate `order_id` rows found: **{duplicate_order_rows:,}**.

Missing `payment_value` records: **{missing_payment_values:,}**.

**Conclusion:**  
The order-level master dataset protects the analysis from many-to-many merge errors. This is essential because incorrect joins between payments and items can inflate Gross Payment Value and produce wrong KPIs.
"""))



# ✅ Final EDA Conclusions

## 1. Revenue Performance

The marketplace generated **16.01M** in total revenue across **99,441 unique orders**.

The average order value was **161**.

The strongest revenue month was **2017-11**, with **1.19M** revenue from **7,544 orders**.

**Conclusion:**  
Revenue performance is not evenly distributed across time. The best-performing month, **2017-11**, should be studied further to understand whether it was driven by seasonality, growth, campaigns, or category mix.

---

## 2. Category Concentration

The highest revenue category was **health_beauty**, generating **1.44M**.

The top 5 categories generated **38.8%** of total revenue.  
The top 10 categories generated **61.8%** of total revenue.

About **17 categories** are needed to reach roughly 80% of category revenue.

**Conclusion:**  
Revenue is concentrated in a limited set of product categories. The business should prioritize these categories for marketing, inventory planning, and seller partnerships, while also managing concentration risk.

---

## 3. Customer Retention

The dataset contains **96,096 unique customers**.

- **93,099 customers** purchased only once.
- **2,997 customers** purchased more than once.
- The one-time customer rate is **96.9%**.
- The repeat customer rate is **3.1%**.
- Average orders per customer is **1.03**.

**Conclusion:**  
The marketplace has a clear retention opportunity. Since **96.9%** of customers purchased only once, campaigns such as remarketing, email follow-ups, loyalty offers, and personalized recommendations could improve repeat purchasing.

---

## 4. Regional Revenue

The strongest customer state was **SP**, generating **6.00M**, or **37.5%** of total revenue.

**Conclusion:**  
Revenue is geographically concentrated. The business should understand what makes **SP** strong and use those learnings for other regions.

---

## 5. Payment Behavior

The most used payment method was **credit_card**, representing **76.6%** of orders.

**Conclusion:**  
The dominant payment method should be reliable because checkout issues in this method could affect a large share of orders.

---

## 6. Delivery Performance

The late delivery rate was **6.8%**.

Average delivery delay was **-11.9 days** compared with the estimated delivery date.

Average delivery time was **12.1 days** from purchase to customer delivery.

**Conclusion:**  
Delivery performance is a key operational area. Late delivery can damage customer satisfaction and should be monitored by seller, customer state, and product category.

---

## 7. Seller Concentration

The top seller was **4869f7a5dfa277a7dca6462dcf3b52b2**, generating **253.1K**.

The top 10 sellers contributed **12.7%** of total revenue.

About **563 sellers** are needed to reach roughly 80% of seller revenue.

**Conclusion:**  
High-performing sellers are important strategic partners. If revenue depends heavily on a small number of sellers, the marketplace should manage dependency risk and support seller quality.

---

## 8. Payment Value Distribution

The median payment value was **105**.

The 95th percentile payment value was **453**.

The maximum payment value was **13.7K**.

**Conclusion:**  
Payment values are skewed, so average order value should not be interpreted alone. Median and percentile values provide a better view of typical customer spend.

---

## 9. Data Quality and Data Grain

The final dataset contains **99,441 rows** and **99,441 unique orders**.

Duplicate `order_id` rows found: **0**.

Missing `payment_value` records: **0**.

**Conclusion:**  
The order-level master dataset protects the analysis from many-to-many merge errors. This is essential because incorrect joins between payments and items can inflate revenue and produce wrong KPIs.


# 21. Final Learning Note

The notebook uses this pattern again and again:

```text
1. Select the right columns
2. Group the data
3. Calculate business numbers
4. Build the chart
5. Format the chart
6. Write a conclusion using actual values
```

The most important professional lesson is:

> Good EDA is not only about good-looking charts. It is about correct calculations, correct data grain, and clear business interpretation.
